In [2]:
from fenics import *
import numpy as np

# Classe para mapear IDs da malha para valores de propriedades
class PropriedadeExpression(UserExpression):
    def __init__(self, subdomains, values, **kwargs):
        super().__init__(**kwargs)
        self.subdomains = subdomains
        self.values = values
    
    def eval_cell(self, values, x, cell):
        subdomain_id = self.subdomains[cell.index]
        values[0] = self.values.get(subdomain_id, self.values[1])

    def value_shape(self):
        return ()

# Classe para definir a fonte de calor do laser
class QrExpression(UserExpression):
    def __init__(self, pontos_laser, A_val, r0, **kwargs):
        super().__init__(**kwargs)
        self.pontos_laser = pontos_laser
        self.A_val = A_val
        self.r0 = r0

    def eval(self, values, x):
        total_qr = 0
        for x0, y0 in self.pontos_laser:
            r2 = (x[0] - x0)**2 + (x[1] - y0)**2
            total_qr += self.A_val * np.exp(-r2 / self.r0**2)
        values[0] = total_qr
    
    def value_shape(self):
        return ()

# Definição dos parâmetros do problema
k_saudavel, k_tumoral = 0.5, 0.55  # condutividade térmica
wb_saudavel, wb_tumoral = 0.00051, 0.00125  # perfusão sanguínea
Qm_saudavel, Qm_tumoral = 420.0, 4200.0  # calor metabólico
rhob = 1000.0  # densidade do sangue
cb = 4200.0  # calor específico do sangue
Ta = 37.0  # temperatura arterial
r0 = 3.1e-3  # raio de espalhamento do ponto de atuação do laser

# Carregando a malha e definindo as condições
mesh = Mesh("malha.xml")
V = FunctionSpace(mesh, "Lagrange", 1)
subdomains = MeshFunction("size_t", mesh, "malha_physical_region.xml")

# Mapeando os IDs das sub-regiões para as propriedades físicas
k = PropriedadeExpression(subdomains, {1: k_saudavel, 2: k_tumoral})
wb = PropriedadeExpression(subdomains, {1: wb_saudavel, 2: wb_tumoral})
Qm = PropriedadeExpression(subdomains, {1: Qm_saudavel, 2: Qm_tumoral})

dx = Measure('dx', domain=mesh, subdomain_data=subdomains)

# Condição de Dirichlet na fronteira esquerda para x = 0
class LeftBoundary(SubDomain):
    def inside(self, x, on_boundary):
        tol = 1e-14
        return on_boundary and np.abs(x[0]) < tol

left_boundary = LeftBoundary()
bc = DirichletBC(V, Constant(Ta), left_boundary)

T = TrialFunction(V)
v = TestFunction(V)

# Função para executar as simulações
def run_simulation(T, v, dx, bc, pontos_laser, A_val, output_filename):
    a = k * dot(grad(T), grad(v)) * dx + rhob * cb * wb * T * v * dx
    L = (rhob * cb * wb * Ta + Qm + QrExpression(pontos_laser, A_val, r0)) * v * dx
    
    T = Function(V)
    solve(a == L, T, bc)

    print(f"\nSimulação Concluída: salvando arquivo {output_filename}...")
    print(f"Temperatura Mínima: {T.vector().min():.2f} °C")
    print(f"Temperatura Máxima: {T.vector().max():.2f} °C")
    
    vtkfile = File(output_filename)
    vtkfile << T

# Executa a simulação com 1 ponto de injeção
ponto_laser_1p = [(0.05, 0.05)]
A_1ponto = 1.3e6
run_simulation(T, v, dx, bc, ponto_laser_1p, A_1ponto, 'solucao_um_ponto.pvd')

# Executa simulação com 4 pontos de injeção
pontos_laser_4p = [(0.045, 0.045), (0.055, 0.045), (0.045, 0.055), (0.055, 0.055)]
A_4pontos = 0.325e6
run_simulation(T, v, dx, bc, pontos_laser_4p, A_4pontos, 'solucao_quatro_pontos.pvd')

Solving linear variational problem.

Simulação Concluída: salvando arquivo solucao_um_ponto.pvd...
Temperatura Mínima: 37.00 °C
Temperatura Máxima: 56.31 °C
Solving linear variational problem.

Simulação Concluída: salvando arquivo solucao_quatro_pontos.pvd...
Temperatura Mínima: 37.00 °C
Temperatura Máxima: 46.85 °C
